# Read In, Clean, and Merge Data

## Read in the data

### Read in the gated station entry data

In [35]:
# ALC
# Imports

import pandas as pd 
from os import walk
import re

In [36]:
# ALC
# specify input files. we'll start with 2022-2025 but we may narrow this down later
input_file_2022 = "../data/gated station entries/GSE_2022.csv"
input_file_2023 = "../data/gated station entries/GSE_2023.csv"
input_file_2024 = "../data/gated station entries/GSE_2024.csv"
input_file_2025 = "../data/gated station entries/GSE_2025.csv"

gse_2022 = pd.read_csv(input_file_2022)
gse_2023 = pd.read_csv(input_file_2023)
gse_2024 = pd.read_csv(input_file_2024)
gse_2025 = pd.read_csv(input_file_2025)

station_entries = pd.concat([gse_2022, gse_2023, gse_2024, gse_2025])

print(station_entries.shape)

(3731053, 7)


### Read in the service alerts data

In [37]:
# ALC
# these are as downloaded from the MBTA's Open Data Portal. There is one file
# for every month of the year. 
# todo see if we can just use the mbta api
input_filenames = [[2022, ['alerts_2022_01', 'alerts_2022_02', 'alerts_2022_03', 
                           'alerts_2022_04', 'alerts_2022_05', 'alerts_2022_06', 
                           'alerts_2022_07', 'alerts_2022_08', 'alerts_2022_09', 
                           'alerts_2022_10', 'alerts_2022_11', 'alerts_2022_12']],
                   [2023, ['alerts_2023_01', 'alerts_2023_02', 'alerts_2023_03', 
                           'alerts_2023_04', 'alerts_2023_05', 'alerts_2023_06', 
                           'alerts_2023_07', 'alerts_2023_08', 'alerts_2023_09', 
                           'alerts_2023_10', 'alerts_2023_11', 'alerts_2023_12']],
                   [2024, ['2024-01_ALERTS', '2024-02_ALERTS', '2024-03_ALERTS', 
                           '2024-04_ALERTS', '2024-05_ALERTS', '2024-06_ALERTS', 
                           '2024-07_ALERTS', '2024-08_ALERTS', '2024-09_ALERTS', 
                           '2024-10_ALERTS','2024-11_ALERTS', '2024-12_ALERTS']],
                   [2025, ['2025-01_ALERTS', '2025-02_ALERTS', '2025-03_ALERTS', 
                           '2025-04_ALERTS', '2025-05_ALERTS', '2025-06_ALERTS']]]
# initialize array
alerts = []

for year, filenames in input_filenames:
    for filename in filenames:
        path = f'../data/service alerts/{year}/{filename}.csv'
        # including low_memory = false because we have some nas in columns
        month_alerts = pd.read_csv(path, quotechar='"', low_memory = False)
        alerts.append(month_alerts)
        
# put it all together
# this is HUGE when you keep every service alert type
service_alerts = pd.concat(alerts)
print(service_alerts.shape)

(2596379, 27)


### Read in the speed restrictions data

In [38]:
#ALC

# todo see if we can just use the mbta api
slow_zone_files = []
# found this after doing service alerts above
# will focus more on switching to use the api than making consistent for now
for (root, dirs, filenames) in walk("../data/slow zones/"):
    slow_zone_files.extend(filenames)
    break
    
slow_zone_data = []

for file in slow_zone_files:
    path = f"../data/slow zones/{file}"
    if (re.search("\.csv$", file)):
        sz = pd.read_csv(path, quotechar='"')
        slow_zone_data.append(sz)
        
slow_zones = pd.concat(slow_zone_data)
slow_zones.shape

(127922, 32)

### Read in the commuter rail ridership data

In [39]:
# ALC
cr_ridership_input = "../data/commuter rail ridership/MBTA_Commuter_Rail_Ridership_by_Service_Date_and_Line.csv"
cr_ridership = pd.read_csv(cr_ridership_input)

cr_ridership.shape

(20254, 4)

### Read in the reliability data

In [40]:
# ALC
reliability_input = "../data/reliability/MBTA Bus, Commuter Rail, & Rapid Transit Reliability.csv"
reliability = pd.read_csv(reliability_input)

reliability.shape

(963838, 13)

## Clean the data and prepare for join

### Make date format consistent

In [52]:
#ALC
# the date in the reliability data appears yyyy/mm/dd hh:mm:ss+00
# we will use yyyy-mm-dd for service dates
reliability['service_date'] = pd.to_datetime(reliability['service_date']).dt.date

# cr_ridership also uses yyyy/mm/dd hh:mm:ss+00
cr_ridership['service_date'] = pd.to_datetime(cr_ridership['service_date']).dt.date

# slow zones uses what we expect but we just rename the column for consistency
slow_zones = slow_zones.rename(columns = {'Calendar_Date': 'service_date'})

# service_alerts is more complicated. there appear to be some problem
# rows. for example in the june 2025 dataset there is a detour alert from july
# of 2024 that never closed
service_alerts['active_period_start_date'] = pd.to_datetime(service_alerts['active_period_start_date']).dt.date
service_alerts['active_period_start_dt'] = pd.to_datetime(service_alerts['active_period_start_dt']).dt.date
service_alerts['active_period_end_dt'] = pd.to_datetime(service_alerts['active_period_end_dt']).dt.date
service_alerts['created_dt'] = pd.to_datetime(service_alerts['created_dt']).dt.date

# gated station entries are given by the half hour. we will need to
# aggregate these down to the day
station_entries = station_entries.groupby(['service_date', 'route_or_line'], as_index=False)['gated_entries'].sum()

0    2022-01-01
1    2022-01-01
2    2022-01-01
3    2022-01-01
4    2022-01-01
Name: service_date, dtype: object

## Join the data